# Validate primary keys with DQX
Reads the primary key registry and checks key uniqueness and null key values for every active table using Databricks Labs DQX, writing results to the central results table.

In [0]:
%pip install databricks-labs-dqx

In [0]:
dbutils.library.restartPython()

In [0]:
# DQX-based primary key validation: returns the same metrics as the Python-native runner
from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient

dq_engine = DQEngine(WorkspaceClient())

def validate_primary_key_dqx(table_name, key_columns):
    df = spark.table(table_name)

    unique_check = [{"criticality": "error",
                     "check": {"function": "is_unique", "arguments": {"columns": key_columns}}}]
    not_null_check = [{"criticality": "error",
                       "check": {"function": "is_not_null", "for_each_column": key_columns}}]

    # Apply each check separately, so failures can be attributed to the right metric
    _, duplicate_rows = dq_engine.apply_checks_by_metadata_and_split(df, unique_check)
    _, null_rows = dq_engine.apply_checks_by_metadata_and_split(df, not_null_check)

    row_count = df.count()
    null_key_count = null_rows.count()

    # DQX flags every row sharing a key; convert to extra rows to match the results table definition
    flagged_rows = duplicate_rows.count()
    flagged_keys = duplicate_rows.select(*key_columns).distinct().count()
    duplicate_count = flagged_rows - flagged_keys

    distinct_key_count = row_count - null_key_count - duplicate_count

    test_result = "PASS" if null_key_count == 0 and duplicate_count == 0 else "FAIL"

    return {
        "table_name": table_name,
        "key_columns": key_columns,
        "row_count": row_count,
        "distinct_key_count": distinct_key_count,
        "null_key_count": null_key_count,
        "duplicate_count": duplicate_count,
        "test_result": test_result,
        "error_message": None
    }

In [0]:
# Validate every active registered table; record errors and continue
import pandas as pd

registry = (spark.table("validation_sweep.metadata.pk_registry")
                 .where("is_active")
                 .collect())

results = []

for r in registry:
    full_name = f"{r.catalog_name}.{r.schema_name}.{r.table_name}"
    try:
        results.append(validate_primary_key_dqx(full_name, list(r.key_columns)))
    except Exception as e:
        results.append({
            "table_name": full_name,
            "key_columns": list(r.key_columns),
            "test_result": "ERROR",
            "error_message": str(e).split("\n")[0][:500]
        })

columns = ["table_name", "key_columns", "test_result",
           "row_count", "distinct_key_count", "null_key_count", "duplicate_count",
           "error_message"]

display(pd.DataFrame(results, columns=columns))

In [0]:
# Append this run's results to the central results table
import uuid
import datetime

results_table = "validation_sweep.results.validation_results"
method = "dqx"
test_type = "pk_integrity"
run_id = str(uuid.uuid4())
run_ts = datetime.datetime.now(datetime.timezone.utc)

run_info = {
    "run_id": run_id,
    "run_ts": run_ts,
    "run_date": run_ts.date(),
    "method": method,
    "test_type": test_type,
}

target_schema = spark.table(results_table).schema
column_names = target_schema.fieldNames()
rows = []

for r in results:
    key_columns_str = ",".join(str(c) for c in r["key_columns"])
    check_id = "|".join([method, test_type, str(r["table_name"]), key_columns_str])

    row = {**r, **run_info, "check_id": check_id}
    rows.append(tuple(row.get(c) for c in column_names))

if rows:
    results_df = spark.createDataFrame(rows, schema=target_schema)
    results_df.write.mode("append").saveAsTable(results_table)
    print(f"Wrote {len(rows)} results for run {run_id}")
else:
    print("No validation results to write")